# Dataset Preparation for our approach

 ##### ⚠️ WARNING ⚠️
This notebook contains a proof of concept that generates synthetic data using Large Language Models (LLMs).

**IMPORTANT DISCLAIMERS:**
* This approach is provided WITHOUT ANY WARRANTY, express or implied
* NO verification of data quality or accuracy is guaranteed
* The generated data may contain biases, inaccuracies, or fabricated information
* This synthetic data should NOT be used for any production systems
* You are NOT ALLOWED to use this for any real-world applications
* Any use of this synthetic data is entirely at your own risk

By proceeding, you acknowledge these limitations and agree to use this only for experimental purposes.

# Importing the necessary libraries

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from datasets import load_dataset
import json
import time
import random
from tqdm.auto import tqdm

 # API Configuration

In [ ]:

load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_KEY", "YOUR_OPENAI_API_KEY")

if OPENAI_API_KEY == "YOUR_OPENAI_API_KEY":
     print("Warning: Replace 'YOUR_OPENAI_API_KEY' with your actual key or set the OPENAI_API_KEY environment variable.")
model_name = "gpt-3.5-turbo" 

client = OpenAI(api_key=OPENAI_API_KEY)
print(f"OpenAI API configured with model: {model_name}")

# Define the base prompt template

In [ ]:

prompt_template = """You are an AI assistant specializing in generating structured therapy plans. Read the following user's mental health concern and the multiple counselor responses to it (separated by ---SEPARATOR---). Based on **both** the user's concern and all counselor responses, generate a structured therapy plan. Aim to create a plan that addresses the user's issues and is informed by the collective counselors' perspectives or suggestions.

Format the output strictly as follows, with each section clearly labeled:

Core Issue: [Summarize the main problem(s) based on user text and collective counselor insights]
Suggested Coping Strategies: [List potential strategies mentioned or implied across the counselor responses, or generally relevant ones based on the user text and combined insights]
Immediate Actions: [Suggest immediate steps based on user text and collective counselor insights]
Long-term Focus Areas: [Suggest areas for deeper work based on user text and collective counselor insights]

If a section cannot be determined or if the combined text is not suitable for generating a plan, indicate 'Not specified'.

User Concern:
---
{user_concern}
---

Counselor Responses (responses from multiple counselors separated by ---SEPARATOR---):
---
{counselor_response}
---

Structured Therapy Plan:
"""

required_keys = ["Core Issue", "Suggested Coping Strategies", "Immediate Actions", "Long-term Focus Areas"]
failure_indicators = ['Not specified', '[Summarize the main problem(s) based on user text and all counselor responses]',
                      '[List potential strategies mentioned, implied, or generally relevant ones synthesized from all texts]',
                      '[Suggest immediate steps the user could take or consider based on all texts]',
                      '[Suggest areas for deeper work in therapy based on all texts]',
                      '']

print("Prompt template and filters defined.")

# 

#  Loading dataset


In [ ]:

print("Loading dataset...")
dataset = load_dataset("nbertagnolli/counsel-chat")
train_data = dataset['train']
print(f"Dataset loaded. Total rows: {len(train_data)}")

print("Grouping data by questionID and combining answerText...")

grouped_data = {}
for row in train_data:
    question_id = row['questionID']
    if question_id not in grouped_data:
        grouped_data[question_id] = {
            'questionID': question_id,
            'questionText': row['questionText'],
            'answerTexts': []
        }
    if row['answerText'] is not None:
        grouped_data[question_id]['answerTexts'].append(row['answerText'])


processed_data = []
for question_id, data in grouped_data.items():
    if data['answerTexts']:
       
        combined_answer_text = " ---SEPARATOR--- ".join([text for text in data['answerTexts'] if text is not None])
    else:
        combined_answer_text = "" 

    processed_data.append({
        'questionID': data['questionID'],
        'questionText': data['questionText'],
        'combined_answerText': combined_answer_text
    })

print(f"Data processed. Found {len(processed_data)} unique questions.")

print("\nFirst processed entry:")
# import pprint
# pprint.pprint(processed_data[0]) 

# Data Generation Loop


In [ ]:
import time

In [ ]:
generated_data_raw_hybrid = []
output_filename = "counselchat_generated_structured_plans_hybrid_gpt35.jsonl" # Filename

print(f"Starting data generation for {len(processed_data)} rows using {model_name}...")
print("Consult OpenAI documentation for API limits and pricing.")
print("A progress bar will show the processing status.")

for i, example in tqdm(enumerate(processed_data), total=len(processed_data), desc="Generating Plans"):
    user_concern = example['questionText']
    counselor_response = example['combined_answerText'] 


    full_prompt = prompt_template.format(
        user_concern=user_concern,
        counselor_response=counselor_response
    )

    generated_text = None
    structured_plan = None
    is_suitable_for_finetuning = False
    error_message = None

    max_retries = 5
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model_name,
                messages=[
                    {"role": "system", "content": "You are an AI assistant specializing in generating structured therapy plans."},
                    {"role": "user", "content": full_prompt}
                ],
                temperature=0.7, 
                max_tokens=500 
            )
            generated_text = response.choices[0].message.content

            break 

        except Exception as e:
            error_message = str(e)
            print(f"\nAPI Error on row {i+1}, attempt {attempt+1}/{max_retries}: {error_message}")

            if e.code == "rate_limit_exceeded":
                 print(f"Rate limit hit on row {i+1}. Pausing...")
                 time.sleep(60) 
            elif attempt < max_retries - 1:
                time.sleep(2 ** attempt + random.uniform(0, 1)) 
            else:
                print(f"\nMax retries reached for row {i+1}. Skipping this row after persistent errors.")



    if generated_text:
        structured_plan = {}
        lines = generated_text.split('\n')
        current_key = None
        current_value = []

        for line in lines:
            line = line.strip()
            if line.startswith("Core Issue:"):
                if current_key: structured_plan[current_key] = "\n".join(current_value).strip()
                current_key = "Core Issue"
                current_value = [line[len("Core Issue:"):].strip()]
            elif line.startswith("Suggested Coping Strategies:"):
                if current_key: structured_plan[current_key] = "\n".join(current_value).strip()
                current_key = "Suggested Coping Strategies"
                current_value = [line[len("Suggested Coping Strategies:"):].strip()]
            elif line.startswith("Immediate Actions:"):
                if current_key: structured_plan[current_key] = "\n".join(current_value).strip()
                current_key = "Immediate Actions"
                current_value = [line[len("Immediate Actions:"):].strip()]
            elif line.startswith("Long-term Focus Areas:"):
                if current_key: structured_plan[current_key] = "\n".join(current_value).strip()
                current_key = "Long-term Focus Areas"
                current_value = [line[len("Long-term Focus Areas:"):].strip()]
            else:
                current_value.append(line)

        if current_key:
             structured_plan[current_key] = "\n".join(current_value).strip()

        if all(key in structured_plan for key in required_keys):
            if all(structured_plan[key] not in failure_indicators for key in required_keys):
                 is_suitable_for_finetuning = True


    row_data = {
        'questionID': example['questionID'],
        'original_questionText': user_concern,
        'original_combined_answerText': counselor_response,  
        'generated_structured_plan': structured_plan,
        'is_suitable_for_finetuning': is_suitable_for_finetuning,
        'error': error_message
    }
    generated_data_raw_hybrid.append(row_data)

    # if (i + 1) % 50 == 0:
    #      print(f"Processed {i+1} rows.")

    # if (i + 1) % 10 == 0: 
    #     try:
    #         with open(output_filename, 'w', encoding='utf-8') as f:
    #             for entry in generated_data_raw_hybrid:
    #                 json.dump(entry, f, ensure_ascii=False)
    #     except Exception as e:
    #         print(f"Error saving data: {e}")

# Saving

In [ ]:

output_filename = "counselchat_generated_structured_plans_hybrid_gpt35.jsonl"

print(f"Saving final results to {output_filename}...")

try:
    with open(output_filename, 'w', encoding='utf-8') as f:
        for entry in generated_data_raw_hybrid:
            json.dump(entry, f, ensure_ascii=False)
            f.write('\n')

    print(f"Raw generated data saved to {output_filename}.")

    suitable_candidates_count = sum(1 for entry in generated_data_raw_hybrid if entry['is_suitable_for_finetuning'])
    print(f"Initial suitable candidates (before manual review): {suitable_candidates_count} out of {len(train_data)}")

    print("\nNext Step: Manually review the generated data in the output file to select the highest quality examples for fine-tuning.")
    print(f"The file {output_filename} contains the generated data.")

except Exception as e:
    print(f"Error saving final results: {e}")

# Let's use some preprocessing

In [ ]:
from preprocess_chats import preprocess_pipeline
input_file = "counselchat_generated_structured_plans_hybrid_gpt35.jsonl"

results = preprocess_pipeline(
    input_file=input_file
)